In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "tennie2019chimpanzees")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Tennie_2019_10329_2019_754_MOESM1_ESM.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)


df['study_id']="tennie2019chimpanzees"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
# df.columns


In [3]:
df.rename(columns={"subject_name": "ape",
    "session 1 or 2": "session",
    "trial_in_session":"trial",
    "causal button pressed 1=yes, 0=no": "causal_button_pressed",
    "trial_no across sessions": "trial_number_across_sessions",
    "causal side (l or r)": "causal side-l_or_r",
    "if button pressed: l or r": "button_pressed-l_or_r"}, inplace=True)

In [4]:
code_list=["causal_button_pressed"]
for index, x in enumerate(code_list):    
    df[x] = df[x].astype(str)
    temp=[]
    for entry in df[x]:
        if entry == '0':
            entry = "no"
        elif entry =='1':
            entry = "yes"
        temp.append(entry)
    df = df.assign(temp_col=temp)
    df=df.rename(columns={'temp_col': x+'_codes'})

In [5]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

In [6]:
# df.columns
df.rename(columns={"ape": "participant",
                   'causal side-l_or_r':'causal_side-l_or_r'}, inplace=True)

In [7]:
tennie2019chimpanzees_standardized=df[['study_id', 'participant', 'sex','species', 'session', 'trial', 'trial_number_across_sessions',
       'causal_button_pressed', 'causal_button_pressed_codes',  'causal_side-l_or_r',
       'button_pressed-l_or_r']]
comp_out_path_stand = os.path.join(out_pathway, 'tennie2019chimpanzees_standardized.csv')
tennie2019chimpanzees_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

names =tennie2019chimpanzees_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
tennie2019chimpanzees_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'tennie2019chimpanzees_glossary.csv')
tennie2019chimpanzees_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

